# Section 3 — Error in Euler's Method

[← Euler's Method](02_euler_method.ipynb) · [Module 3](README.md) · [Next: Taylor Methods →](04_taylor_methods.ipynb)

Euler's method is first-order accurate globally. This notebook separates two related ideas:

- the error introduced in a **single step**;
- the accumulated **global error** after many steps.

### A terminology note

Two conventions are common. The one-step defect

$$
y(t_{n+1})-\bigl[y(t_n)+h f(t_n,y(t_n))\bigr]
$$

is $O(h^2)$. If that quantity is divided by $h$, the resulting scaled local truncation error is $O(h)$. We will state explicitly which convention is being used.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Test problem

Consider

$$
y'=y-t^2+1,\qquad y(0)=0.5,\qquad 0\le t\le2.
$$

The exact solution is

$$
y(t)=(t+1)^2-\frac12 e^t.
$$

In [2]:
def f(t, y):
    return y - t**2 + 1


def exact(t):
    return (t + 1)**2 - 0.5*np.exp(t)


def euler(f, t0, y0, h, n_steps):
    t = np.empty(n_steps + 1)
    w = np.empty(n_steps + 1)
    t[0], w[0] = t0, y0
    for n in range(n_steps):
        w[n + 1] = w[n] + h*f(t[n], w[n])
        t[n + 1] = t[n] + h
    return t, w

## 2. Measure the global error

For several step sizes, compute the error at $t=2$. If Euler's method is first-order globally, halving $h$ should eventually reduce the error by approximately a factor of 2.

In [3]:
step_sizes = np.array([0.4, 0.2, 0.1, 0.05, 0.025])
errors = []

print("     h    steps       w(2)          exact          error      observed p")
previous_error = None
for h in step_sizes:
    n_steps = round(2/h)
    t, w = euler(f, 0.0, 0.5, h, n_steps)
    error = abs(exact(2.0) - w[-1])
    errors.append(error)
    if previous_error is None:
        p_text = "   --"
    else:
        p = np.log(previous_error/error)/np.log(2)
        p_text = f"{p:7.4f}"
    print(f"{h:7.3f}   {n_steps:4d}   {w[-1]:12.8f}   {exact(2.0):12.8f}   {error:10.6f}   {p_text}")
    previous_error = error

errors = np.array(errors)

     h    steps       w(2)          exact          error      observed p
  0.400      5     4.55958400     5.30547195     0.745888      --
  0.200     10     4.86578450     5.30547195     0.439687    0.7625
  0.100     20     5.06350003     5.30547195     0.241972    0.8616
  0.050     40     5.17800621     5.30547195     0.127466    0.9247
  0.025     80     5.23997690     5.30547195     0.065495    0.9607


In [4]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(step_sizes, errors, marker="o", label="Euler error at t=2")

# First-order reference line, scaled to pass through the last data point.
reference = errors[-1] * (step_sizes / step_sizes[-1])
ax.loglog(step_sizes, reference, linestyle="--", label=r"reference: $O(h)$")

ax.set_xlabel("step size h")
ax.set_ylabel("absolute error")
ax.set_title("Euler's method: global error versus step size")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
plt.show()

## 3. The theoretical global error bound

Suppose $f(t,y)$ satisfies a Lipschitz condition in $y$ with constant $L$, and $|y''(t)|\le M$. A standard Euler bound is

$$
|y(t_i)-w_i|\le \frac{Mh}{2L}\left(e^{L(t_i-a)}-1\right).
$$

For this problem, $f_y=1$, so we can take $L=1$. Also

$$
y''(t)=2-\frac12e^t.
$$

On $[0,2]$, we can bound $|y''(t)|$ by checking the endpoints because $y''$ is decreasing.

In [5]:
L = 1.0
M = max(abs(2 - 0.5*np.exp(0)), abs(2 - 0.5*np.exp(2)))
h = 0.2
bound = (M*h/(2*L)) * (np.exp(L*2) - 1)

t, w = euler(f, 0.0, 0.5, h, round(2/h))
actual_error = abs(exact(2.0) - w[-1])

print(f"L = {L:.1f}")
print(f"M = {M:.6f}")
print(f"Actual error at t=2 with h=0.2: {actual_error:.6f}")
print(f"Theoretical upper bound:             {bound:.6f}")
print(f"Bound / actual error:                {bound/actual_error:.2f}")

L = 1.0
M = 1.694528
Actual error at t=2 with h=0.2: 0.439687
Theoretical upper bound:             1.082643
Bound / actual error:                2.46


The bound guarantees convergence as $h\to0$, but it is often much larger than the observed error. That is normal: a rigorous bound is designed to be safe, not sharp.

## 4. Local error versus global error

Taylor's theorem gives

$$
y(t_{n+1})=y(t_n)+h y'(t_n)+\frac{h^2}{2}y''(\xi_n).
$$

Since $y'=f(t,y)$, one Euler step misses the final term. Therefore:

- **one-step defect:** $O(h^2)$;
- **scaled local truncation error** (divide the defect by $h$): $O(h)$;
- **global error after many Euler steps:** $O(h)$.

The next question is natural: can we keep more of the Taylor expansion and obtain a higher-order method?

[← Euler's Method](02_euler_method.ipynb) · [Next: Taylor Methods →](04_taylor_methods.ipynb)